In [3]:
import pandas as pd

nav = pd.read_csv("../data/raw/nav_history.csv")
print(nav.columns.tolist())

['date', 'scheme_code', 'nav', 'scheme_name']


In [4]:
# ==========================================
# TASK 1: CLEAN NAV_HISTORY.CSV
# ==========================================

import pandas as pd
from pathlib import Path

# File Paths
input_file = Path("../data/raw/nav_history.csv")
output_dir = Path("../data/processed")
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "nav_history.csv"

# ------------------------------------------
# Load Dataset
# ------------------------------------------
nav = pd.read_csv(input_file)

print("=" * 60)
print("ORIGINAL DATASET INFO")
print("=" * 60)
print("Shape:", nav.shape)

# ------------------------------------------
# Convert Date to Datetime
# ------------------------------------------
nav["date"] = pd.to_datetime(
    nav["date"],
    errors="coerce",
    dayfirst=True
)

# Remove Invalid Dates
invalid_dates = nav["date"].isna().sum()

nav = nav.dropna(subset=["date"])

# ------------------------------------------
# Sort by Scheme Code and Date
# ------------------------------------------
nav = nav.sort_values(
    by=["scheme_code", "date"]
)

# ------------------------------------------
# Remove Duplicate Rows
# ------------------------------------------
duplicates_before = nav.duplicated().sum()

nav = nav.drop_duplicates()

duplicates_removed = duplicates_before

# ------------------------------------------
# Forward Fill Missing NAV
# ------------------------------------------
missing_nav_before = nav["nav"].isnull().sum()

nav["nav"] = (
    nav.groupby("scheme_code")["nav"]
       .ffill()
)

missing_nav_after = nav["nav"].isnull().sum()

# ------------------------------------------
# Validate NAV > 0
# ------------------------------------------
invalid_nav_count = len(
    nav[nav["nav"] <= 0]
)

# Remove Invalid NAV Records
nav = nav[
    nav["nav"] > 0
]

# ------------------------------------------
# Final Sorting
# ------------------------------------------
nav = nav.sort_values(
    by=["scheme_code", "date"]
)

# ------------------------------------------
# Save Cleaned File
# ------------------------------------------
nav.to_csv(
    output_file,
    index=False
)

# ------------------------------------------
# Summary
# ------------------------------------------
print("\n" + "=" * 60)
print("DATA CLEANING SUMMARY")
print("=" * 60)

print(f"Invalid Dates Removed     : {invalid_dates}")
print(f"Duplicates Removed        : {duplicates_removed}")
print(f"Missing NAV Before FFill  : {missing_nav_before}")
print(f"Missing NAV After FFill   : {missing_nav_after}")
print(f"Invalid NAV Removed       : {invalid_nav_count}")

print("\nFinal Shape:")
print(nav.shape)

print("\nCleaned file saved at:")
print(output_file)

print("\nTask 1 Completed Successfully!")

ORIGINAL DATASET INFO
Shape: (19560, 4)

DATA CLEANING SUMMARY
Invalid Dates Removed     : 11865
Duplicates Removed        : 0
Missing NAV Before FFill  : 0
Missing NAV After FFill   : 0
Invalid NAV Removed       : 0

Final Shape:
(7695, 4)

Cleaned file saved at:
..\data\processed\nav_history.csv

Task 1 Completed Successfully!


In [6]:
# ==========================================
# TASK 2: CLEAN SIP_DATA.CSV
# ==========================================

import pandas as pd
from pathlib import Path

# File Paths
input_file = Path("../data/raw/sip_data.csv")
output_dir = Path("../data/processed")
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "sip_data.csv"

# ------------------------------------------
# Load Dataset
# ------------------------------------------
sip = pd.read_csv(input_file)

print("=" * 60)
print("ORIGINAL DATASET INFO")
print("=" * 60)
print("Shape:", sip.shape)

# ------------------------------------------
# Convert Month Column to Datetime
# ------------------------------------------
sip["month"] = pd.to_datetime(
    sip["month"],
    errors="coerce"
)

# Remove invalid dates
invalid_dates = sip["month"].isna().sum()

sip = sip.dropna(subset=["month"])

# ------------------------------------------
# Remove Duplicate Records
# ------------------------------------------
duplicates_before = sip.duplicated().sum()

sip = sip.drop_duplicates()

duplicates_removed = duplicates_before

# ------------------------------------------
# Numeric Conversion
# ------------------------------------------
numeric_cols = [
    "monthly_sip_amt",
    "units_purchased",
    "total_units",
    "total_invested",
    "current_value",
    "xirr_pct",
    "absolute_return_pct"
]

for col in numeric_cols:
    sip[col] = pd.to_numeric(
        sip[col],
        errors="coerce"
    )

# ------------------------------------------
# Missing Values Check
# ------------------------------------------
missing_before = sip.isnull().sum().sum()

# Fill numeric nulls with 0
sip[numeric_cols] = sip[numeric_cols].fillna(0)

missing_after = sip.isnull().sum().sum()

# ------------------------------------------
# Validate Amount Columns
# ------------------------------------------
invalid_sip_amt = len(
    sip[sip["monthly_sip_amt"] <= 0]
)

sip = sip[
    sip["monthly_sip_amt"] > 0
]

# ------------------------------------------
# Validate Units Purchased
# ------------------------------------------
invalid_units = len(
    sip[sip["units_purchased"] < 0]
)

sip = sip[
    sip["units_purchased"] >= 0
]

# ------------------------------------------
# Validate Total Investment
# ------------------------------------------
invalid_investment = len(
    sip[sip["total_invested"] < 0]
)

sip = sip[
    sip["total_invested"] >= 0
]

# ------------------------------------------
# XIRR Validation
# ------------------------------------------
# Flag unrealistic values

sip["xirr_flag"] = (
    (sip["xirr_pct"] < -100)
    |
    (sip["xirr_pct"] > 200)
)

anomaly_count = sip["xirr_flag"].sum()

# ------------------------------------------
# Sort Data
# ------------------------------------------
sip = sip.sort_values(
    by=["scheme_code", "month"]
)

# ------------------------------------------
# Save Cleaned Dataset
# ------------------------------------------
sip.to_csv(
    output_file,
    index=False
)

# ------------------------------------------
# Summary Report
# ------------------------------------------
print("\n" + "=" * 60)
print("DATA CLEANING SUMMARY")
print("=" * 60)

print(f"Invalid Dates Removed      : {invalid_dates}")
print(f"Duplicates Removed         : {duplicates_removed}")
print(f"Missing Values Before      : {missing_before}")
print(f"Missing Values After       : {missing_after}")
print(f"Invalid SIP Amount Records : {invalid_sip_amt}")
print(f"Invalid Units Records      : {invalid_units}")
print(f"Invalid Investment Records : {invalid_investment}")
print(f"XIRR Anomalies Flagged     : {anomaly_count}")

print("\nFinal Shape:")
print(sip.shape)

print("\nCleaned file saved at:")
print(output_file)

print("\nTask 2 Completed Successfully!")

ORIGINAL DATASET INFO
Shape: (900, 10)

DATA CLEANING SUMMARY
Invalid Dates Removed      : 0
Duplicates Removed         : 0
Missing Values Before      : 0
Missing Values After       : 0
Invalid SIP Amount Records : 0
Invalid Units Records      : 0
Invalid Investment Records : 0
XIRR Anomalies Flagged     : 0

Final Shape:
(900, 11)

Cleaned file saved at:
..\data\processed\sip_data.csv

Task 2 Completed Successfully!


In [8]:
# ==========================================
# TASK 3: CLEAN PERFORMANCE DATASETS
# returns_data.csv
# risk_metrics.csv
# expense_ratio.csv
# ==========================================

import pandas as pd
from pathlib import Path

# ------------------------------------------
# Create Processed Folder
# ------------------------------------------
processed_dir = Path("../data/processed")
processed_dir.mkdir(exist_ok=True)

# ==========================================================
# PART A: CLEAN RETURNS_DATA.CSV
# ==========================================================

print("=" * 60)
print("CLEANING RETURNS_DATA.CSV")
print("=" * 60)

returns = pd.read_csv("../data/raw/returns_data.csv")

original_shape_returns = returns.shape

# Convert return columns to numeric

return_cols = [
    "return_1m_pct",
    "return_3m_pct",
    "return_6m_pct",
    "return_1y_pct",
    "return_3y_pct",
    "return_5y_pct",
    "return_since_inception_pct",
    "category_avg_1y",
    "benchmark_return_1y"
]

for col in return_cols:
    returns[col] = pd.to_numeric(
        returns[col],
        errors="coerce"
    )

# Convert date

returns["as_of_date"] = pd.to_datetime(
    returns["as_of_date"],
    errors="coerce"
)

# Remove duplicates

duplicates_returns = returns.duplicated().sum()

returns = returns.drop_duplicates()

# Flag abnormal returns

returns["return_anomaly"] = False

for col in return_cols:
    returns.loc[
        (returns[col] < -100) |
        (returns[col] > 500),
        "return_anomaly"
    ] = True

return_anomalies = returns["return_anomaly"].sum()

# Save

returns.to_csv(
    processed_dir / "returns_data.csv",
    index=False
)

print("Returns Shape:", original_shape_returns)
print("Duplicate Rows Removed:", duplicates_returns)
print("Return Anomalies:", return_anomalies)

# ==========================================================
# PART B: CLEAN RISK_METRICS.CSV
# ==========================================================

print("\n" + "=" * 60)
print("CLEANING RISK_METRICS.CSV")
print("=" * 60)

risk = pd.read_csv("../data/raw/risk_metrics.csv")

original_shape_risk = risk.shape

risk_cols = [
    "std_dev_1y_pct",
    "std_dev_3y_pct",
    "sharpe_ratio_1y",
    "sharpe_ratio_3y",
    "sortino_ratio",
    "beta",
    "alpha_pct",
    "r_squared",
    "var_95_pct",
    "max_drawdown_pct"
]

for col in risk_cols:
    risk[col] = pd.to_numeric(
        risk[col],
        errors="coerce"
    )

risk["as_of_date"] = pd.to_datetime(
    risk["as_of_date"],
    errors="coerce"
)

duplicates_risk = risk.duplicated().sum()

risk = risk.drop_duplicates()

# Risk anomaly flags

risk["risk_anomaly"] = False

# Beta usually 0–3

risk.loc[
    (risk["beta"] < 0) |
    (risk["beta"] > 3),
    "risk_anomaly"
] = True

# R-squared should be 0–1

risk.loc[
    (risk["r_squared"] < 0) |
    (risk["r_squared"] > 1),
    "risk_anomaly"
] = True

# Std deviation cannot be negative

risk.loc[
    risk["std_dev_1y_pct"] < 0,
    "risk_anomaly"
] = True

risk.loc[
    risk["std_dev_3y_pct"] < 0,
    "risk_anomaly"
] = True

risk_anomalies = risk["risk_anomaly"].sum()

# Save

risk.to_csv(
    processed_dir / "risk_metrics.csv",
    index=False
)

print("Risk Shape:", original_shape_risk)
print("Duplicate Rows Removed:", duplicates_risk)
print("Risk Anomalies:", risk_anomalies)

# ==========================================================
# PART C: CLEAN EXPENSE_RATIO.CSV
# ==========================================================

print("\n" + "=" * 60)
print("CLEANING EXPENSE_RATIO.CSV")
print("=" * 60)

expense = pd.read_csv("../data/raw/expense_ratio.csv")

original_shape_expense = expense.shape

expense_cols = [
    "expense_ratio_direct_pct",
    "expense_ratio_regular_pct",
    "management_fee_pct",
    "other_expenses_pct",
    "sebi_limit_pct"
]

for col in expense_cols:
    expense[col] = pd.to_numeric(
        expense[col],
        errors="coerce"
    )

expense["effective_date"] = pd.to_datetime(
    expense["effective_date"],
    errors="coerce"
)

duplicates_expense = expense.duplicated().sum()

expense = expense.drop_duplicates()

# Expense ratio validation
# Required by assignment:
# 0.1% to 2.5%

expense["expense_ratio_flag"] = False

expense.loc[
    (expense["expense_ratio_direct_pct"] < 0.1) |
    (expense["expense_ratio_direct_pct"] > 2.5),
    "expense_ratio_flag"
] = True

expense.loc[
    (expense["expense_ratio_regular_pct"] < 0.1) |
    (expense["expense_ratio_regular_pct"] > 2.5),
    "expense_ratio_flag"
] = True

expense_anomalies = expense["expense_ratio_flag"].sum()

# Save

expense.to_csv(
    processed_dir / "expense_ratio.csv",
    index=False
)

print("Expense Shape:", original_shape_expense)
print("Duplicate Rows Removed:", duplicates_expense)
print("Expense Ratio Anomalies:", expense_anomalies)

# ==========================================================
# FINAL SUMMARY
# ==========================================================

print("\n" + "=" * 60)
print("TASK 3 COMPLETED SUCCESSFULLY")
print("=" * 60)

print("Processed Files Saved To:")
print("../data/processed/returns_data.csv")
print("../data/processed/risk_metrics.csv")
print("../data/processed/expense_ratio.csv")

CLEANING RETURNS_DATA.CSV
Returns Shape: (15, 13)
Duplicate Rows Removed: 0
Return Anomalies: 0

CLEANING RISK_METRICS.CSV
Risk Shape: (15, 14)
Duplicate Rows Removed: 0
Risk Anomalies: 0

CLEANING EXPENSE_RATIO.CSV
Expense Shape: (15, 10)
Duplicate Rows Removed: 0
Expense Ratio Anomalies: 1

TASK 3 COMPLETED SUCCESSFULLY
Processed Files Saved To:
../data/processed/returns_data.csv
../data/processed/risk_metrics.csv
../data/processed/expense_ratio.csv


In [9]:
# ==========================================
# CLEAN REMAINING 5 DATASETS
# ==========================================

import pandas as pd
from pathlib import Path

# ------------------------------------------
# Paths
# ------------------------------------------

RAW_PATH = Path("../data/raw")
PROCESSED_PATH = Path("../data/processed")

PROCESSED_PATH.mkdir(exist_ok=True)

print("=" * 70)
print("CLEANING REMAINING DATASETS")
print("=" * 70)

# ==========================================================
# 1. FUND_MASTER.CSV
# ==========================================================

print("\nCleaning fund_master.csv...")

fund = pd.read_csv(RAW_PATH / "fund_master.csv")

original_rows = len(fund)

fund = fund.drop_duplicates()

# Convert launch date
fund["launch_date"] = pd.to_datetime(
    fund["launch_date"],
    errors="coerce"
)

# Remove duplicate scheme codes if any
fund = fund.drop_duplicates(subset=["scheme_code"])

fund.to_csv(
    PROCESSED_PATH / "fund_master.csv",
    index=False
)

print(f"Rows Before : {original_rows}")
print(f"Rows After  : {len(fund)}")

# ==========================================================
# 2. SCHEME_DETAILS.CSV
# ==========================================================

print("\nCleaning scheme_details.csv...")

scheme = pd.read_csv(RAW_PATH / "scheme_details.csv")

original_rows = len(scheme)

scheme = scheme.drop_duplicates()

# Convert all date columns automatically
for col in scheme.columns:
    if "date" in col.lower():
        scheme[col] = pd.to_datetime(
            scheme[col],
            errors="coerce"
        )

scheme.to_csv(
    PROCESSED_PATH / "scheme_details.csv",
    index=False
)

print(f"Rows Before : {original_rows}")
print(f"Rows After  : {len(scheme)}")

# ==========================================================
# 3. AUM_DATA.CSV
# ==========================================================

print("\nCleaning aum_data.csv...")

aum = pd.read_csv(RAW_PATH / "aum_data.csv")

original_rows = len(aum)

aum = aum.drop_duplicates()

# Month to datetime
aum["month"] = pd.to_datetime(
    aum["month"],
    errors="coerce"
)

# Numeric columns
aum["aum_cr"] = pd.to_numeric(
    aum["aum_cr"],
    errors="coerce"
)

aum["no_of_folios"] = pd.to_numeric(
    aum["no_of_folios"],
    errors="coerce"
)

# Remove invalid AUM
aum = aum[aum["aum_cr"] > 0]

aum.to_csv(
    PROCESSED_PATH / "aum_data.csv",
    index=False
)

print(f"Rows Before : {original_rows}")
print(f"Rows After  : {len(aum)}")

# ==========================================================
# 4. PORTFOLIO_HOLDINGS.CSV
# ==========================================================

print("\nCleaning portfolio_holdings.csv...")

portfolio = pd.read_csv(
    RAW_PATH / "portfolio_holdings.csv"
)

original_rows = len(portfolio)

portfolio = portfolio.drop_duplicates()

# Numeric columns
portfolio["weight_pct"] = pd.to_numeric(
    portfolio["weight_pct"],
    errors="coerce"
)

portfolio["market_value_cr"] = pd.to_numeric(
    portfolio["market_value_cr"],
    errors="coerce"
)

# Remove negative values
portfolio = portfolio[
    portfolio["weight_pct"] >= 0
]

portfolio = portfolio[
    portfolio["market_value_cr"] >= 0
]

portfolio.to_csv(
    PROCESSED_PATH / "portfolio_holdings.csv",
    index=False
)

print(f"Rows Before : {original_rows}")
print(f"Rows After  : {len(portfolio)}")

# ==========================================================
# 5. BENCHMARK_DATA.CSV
# ==========================================================

print("\nCleaning benchmark_data.csv...")

benchmark = pd.read_csv(
    RAW_PATH / "benchmark_data.csv"
)

original_rows = len(benchmark)

benchmark = benchmark.drop_duplicates()

benchmark["date"] = pd.to_datetime(
    benchmark["date"],
    errors="coerce"
)

numeric_cols = [
    "nifty100",
    "nifty50",
    "nifty100_return_pct",
    "nifty50_return_pct"
]

for col in numeric_cols:
    benchmark[col] = pd.to_numeric(
        benchmark[col],
        errors="coerce"
    )

benchmark.to_csv(
    PROCESSED_PATH / "benchmark_data.csv",
    index=False
)

print(f"Rows Before : {original_rows}")
print(f"Rows After  : {len(benchmark)}")

# ==========================================================
# FINAL SUMMARY
# ==========================================================

print("\n" + "=" * 70)
print("ALL 5 DATASETS CLEANED SUCCESSFULLY")
print("=" * 70)

print("\nProcessed Files Created:")

files = [
    "fund_master.csv",
    "scheme_details.csv",
    "aum_data.csv",
    "portfolio_holdings.csv",
    "benchmark_data.csv"
]

for f in files:
    print(f"✓ {f}")

print("\nLocation:")
print("../data/processed/")

CLEANING REMAINING DATASETS

Cleaning fund_master.csv...
Rows Before : 15
Rows After  : 15

Cleaning scheme_details.csv...
Rows Before : 15
Rows After  : 15

Cleaning aum_data.csv...
Rows Before : 900
Rows After  : 900

Cleaning portfolio_holdings.csv...
Rows Before : 675
Rows After  : 675

Cleaning benchmark_data.csv...
Rows Before : 1304
Rows After  : 1304

ALL 5 DATASETS CLEANED SUCCESSFULLY

Processed Files Created:
✓ fund_master.csv
✓ scheme_details.csv
✓ aum_data.csv
✓ portfolio_holdings.csv
✓ benchmark_data.csv

Location:
../data/processed/
